In [ ]:
import kagglehub
path = kagglehub.dataset_download("manith04/ddd-processed-2-training-frames-type-1")

100%|██████████| 3.37G/3.37G [01:30<00:00, 40.1MB/s]

Extracting files...


In [ ]:
print(path)

/root/.cache/kagglehub/datasets/manith04/ddd-processed-2-training-frames-type-1/versions/1


In [ ]:
!wget https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task

--2026-03-06 15:54:52--  https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.145.207, 74.125.128.207, 74.125.143.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.250.145.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3758596 (3.6M) [application/octet-stream]
Saving to: ‘face_landmarker.task’

face_landmarker.tas 100%[===================>]   3.58M  5.18MB/s    in 0.7s    

2026-03-06 15:54:54 (5.18 MB/s) - ‘face_landmarker.task’ saved [3758596/3758596]



In [ ]:
!uv pip install mediapipe -q

In [ ]:
%%file mark.py

import os
os.environ["MPLBACKEND"] = "agg"

import cv2
import numpy as np
import os

# -----------------------------
# MediaPipe Tasks API
# -----------------------------
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Load model (make sure face_landmarker.task is in same directory)
base_options = python.BaseOptions(model_asset_path="face_landmarker.task")

options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    num_faces=1
)

face_landmarker = vision.FaceLandmarker.create_from_options(options)

# -----------------------------
# Landmark Indices
# -----------------------------
LEFT_EYE_IDX = [33, 160, 158, 133, 153, 144]
RIGHT_EYE_IDX = [263, 387, 385, 362, 373, 380]
MOUTH_IDX = [78, 81, 13, 311, 308, 402, 14, 178]

# -----------------------------
# Helper functions
# -----------------------------
def euclidean_dist(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

def calculate_ear(landmarks, eye_indices):
    p1, p2, p3, p4, p5, p6 = [landmarks[i] for i in eye_indices]
    den = euclidean_dist(p1, p4)

    if den == 0:
      return float("nan")

    ear = (euclidean_dist(p2, p6) + euclidean_dist(p3, p5)) / (2.0 * den)
    return ear

def calculate_mar(landmarks, mouth_indices):
    p11, p12, p13, p14, p17, p18, p19, p20 = [landmarks[i] for i in mouth_indices]
    mar = (
        euclidean_dist(p14, p18)
        + euclidean_dist(p13, p19)
        + euclidean_dist(p12, p20)
    ) / (3.0 * euclidean_dist(p11, p17))
    return mar

# -----------------------------
# Process one image
# -----------------------------
def process_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        return None

    h, w, _ = image.shape

    # Convert BGR → RGB
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Convert to MediaPipe Image
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_image
    )

    # Run detection
    result = face_landmarker.detect(mp_image)

    if result.face_landmarks and len(result.face_landmarks) > 0:
        landmarks = result.face_landmarks[0]

        # Convert normalized → pixel coordinates
        landmark_points = [
            (int(lm.x * w), int(lm.y * h))
            for lm in landmarks
        ]

        left_ear = calculate_ear(landmark_points, LEFT_EYE_IDX)
        right_ear = calculate_ear(landmark_points, RIGHT_EYE_IDX)
        mar = calculate_mar(landmark_points, MOUTH_IDX)

        return left_ear, right_ear, mar

    return None

# -----------------------------
# Process dataset recursively
# -----------------------------
def process_dataset(root_dir, output_root):
    for person_id in os.listdir(root_dir):
        person_path = os.path.join(root_dir, person_id)
        if not os.path.isdir(person_path):
            continue

        for scenario in os.listdir(person_path):
            scenario_path = os.path.join(person_path, scenario)
            if not os.path.isdir(scenario_path):
                continue

            out_scenario_path = os.path.join(output_root, person_id, scenario)
            os.makedirs(out_scenario_path, exist_ok=True)

            for img_file in sorted(os.listdir(scenario_path)):
                if not img_file.lower().endswith((".jpg", ".png")):
                    continue

                img_path = os.path.join(scenario_path, img_file)

                parts = img_file.split('_')
                if len(parts) < 5:
                    continue

                person = parts[0]
                scenario_name = parts[1]
                action = parts[2]
                timestamp = parts[3]

                base_filename = f"{person}_{scenario_name}_{action}"
                output_file = os.path.join(
                    out_scenario_path,
                    f"{base_filename}_EAR_MAR.txt"
                )

                result = process_image(img_path)

                if result:
                    left_ear, right_ear, mar = result
                else:
                    left_ear = right_ear = mar = float('nan')

                with open(output_file, "a") as f:
                    f.write(
                        f"{timestamp},{left_ear:.4f},{right_ear:.4f},{mar:.4f}\n"
                    )

                #print(f"Processed {img_file}")


if __name__ == "__main__":
    root_dataset_dir = "/root/.cache/kagglehub/datasets/manith04/ddd-processed-2-training-frames-type-1/versions/1"
    output_root = "/content"
    process_dataset(root_dataset_dir, output_root)


Writing mark.py


In [ ]:
!uv run mark.py

2026-03-06 16:08:31.602289: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-06 16:08:31.608873: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-06 16:08:31.625760: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772813311.652168    7798 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772813311.659767    7798 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772813311.683187    7798 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin